In [12]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\EC0036AU\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\EC0036AU\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\EC0036AU\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [2]:
df=pd.read_csv('spam.csv',encoding='latin1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df=df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'])
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df.rename(columns={'v1':'target','v2':'text'},inplace=True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
df['target']=encoder.fit_transform(df.target)
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.duplicated().sum()

np.int64(403)

In [7]:
len(df)

5572

In [8]:
df=df.drop_duplicates(keep='first')
len(df)

5169

In [9]:
from nltk.stem.porter import PorterStemmer
import string
ps=PorterStemmer()


In [ ]:
def transform_text(text):
    text=text.lower()
    text=nltk.word_tokenize(text)
    y=[]
    for i in text:
        if i.isalnum():
            y.append(i)
            
    text=y[:]
    y.clear()
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    text=y[:]
    y.clear()
    for i in text:
         y.append(ps.stem(i))
       
    return " ".join(y)

In [14]:
transform_text('Go Unitl jurong point, crazy .. available')

'go unitl jurong point crazi avail'

In [15]:
df['transformed']=df['text'].apply(transform_text)
df.head()

,target,text,transformed
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [16]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfid=TfidfVectorizer(max_features=500)

In [19]:
x=tfid.fit_transform(df['transformed']).toarray()
y=df['target'].values

In [20]:
from sklearn.model_selection import train_test_split
xt,xte,yt,yte=train_test_split(x,y,test_size=0.2,random_state=2)

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier


In [26]:
svc=SVC(kernel='sigmoid',gamma=1)
knc=KNeighborsClassifier()
mnb=MultinomialNB()
dtc=DecisionTreeClassifier(max_depth=5)
lrc=LogisticRegression(solver='liblinear',penalty='l1')
rfc=RandomForestClassifier(n_estimators=50,random_state=2)
abc=AdaBoostClassifier(n_estimators=50,random_state=2)
bc=BaggingClassifier(n_estimators=50,random_state=2)
etc=ExtraTreesClassifier(n_estimators=50,random_state=2)
gbdt=GradientBoostingClassifier(n_estimators=50,random_state=2)
xgb=XGBClassifier(n_estimators=50,random_state=2)

In [34]:
clfs={
    'SVC':svc,
    'KNN':knc,
    'NB':mnb,
    'DT':dtc,
    'LR':lrc,
    'RF':rfc,
    'AdaBoost':abc,
    'Bgc':bc,
    'ETC':etc,
    'GradientBoostingClassifier':gbdt,
    'Xgbclass':xgb
}

In [29]:
from sklearn.metrics import classification_report,accuracy_score,precision_score
def train_classifier(clfs,xt,xte,yt):
    clfs.fit(xt,yt)
    yp=clfs.predict(xte)
    return yp
    

In [59]:
report=[]
for name,i in clfs.items():
    yp=train_classifier(i,xt,xte,yt)
    print(classification_report(yte,yp))
    report.append([name,classification_report(yte,yp)])

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       896
           1       0.93      0.81      0.87       138

    accuracy                           0.97      1034
   macro avg       0.95      0.90      0.92      1034
weighted avg       0.97      0.97      0.97      1034

              precision    recall  f1-score   support

           0       0.92      1.00      0.96       896
           1       1.00      0.46      0.63       138

    accuracy                           0.93      1034
   macro avg       0.96      0.73      0.79      1034
weighted avg       0.93      0.93      0.92      1034

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       896
           1       0.97      0.81      0.88       138

    accuracy                           0.97      1034
   macro avg       0.97      0.90      0.93      1034
weighted avg       0.97      0.97      0.97      1034

              preci

c:\Users\EC0036AU\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\EC0036AU\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.97      0.99      0.98       896
           1       0.94      0.83      0.88       138

    accuracy                           0.97      1034
   macro avg       0.96      0.91      0.93      1034
weighted avg       0.97      0.97      0.97      1034

              precision    recall  f1-score   support

           0       0.93      0.99      0.96       896
           1       0.87      0.50      0.64       138

    accuracy                           0.92      1034
   macro avg       0.90      0.74      0.80      1034
weighted avg       0.92      0.92      0.91      1034

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       896
           1       0.90      0.80      0.85       138

    accuracy                           0.96      1034
   macro avg       0.94      0.90      0.91      1034
weighted avg       0.96      0.96      0.96      1034

              preci

In [62]:
for i in report:
    a,b=i
    print(a)
    print(b)

SVC
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       896
           1       0.93      0.81      0.87       138

    accuracy                           0.97      1034
   macro avg       0.95      0.90      0.92      1034
weighted avg       0.97      0.97      0.97      1034

KNN
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       896
           1       1.00      0.46      0.63       138

    accuracy                           0.93      1034
   macro avg       0.96      0.73      0.79      1034
weighted avg       0.93      0.93      0.92      1034

NB
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       896
           1       0.97      0.81      0.88       138

    accuracy                           0.97      1034
   macro avg       0.97      0.90      0.93      1034
weighted avg       0.97      0.97      0.97      1034

DT
     